# M1 Pro — Token Pipeline on MPS



## Objective (Apple M1 Pro)

Run a simplified LLM inference using the NexusRT Metal backend. Profile
memory residency, tokens/sec, and compare against an MLX baseline.


In [ ]:

import sys; sys.path.insert(0, "..")
from platform_check import assert_apple_silicon
assert_apple_silicon()


In [ ]:

import nexusrt, time
info = nexusrt.firmware.init("m1pro")
print(info)


In [ ]:

# Run a 32-token inference on the Metal backend.
prompt = list(b"Hello, NexusRT on M1 Pro.")
t0 = time.perf_counter()
out = nexusrt.pipeline.run_infer(prompt, max_new_tokens=32)
t1 = time.perf_counter()
text = nexusrt.pipeline.run_postprocess(out)
print(f"Generated {len(out)-len(prompt)} tokens in {(t1-t0)*1e3:.1f} ms "
      f"({(len(out)-len(prompt))/(t1-t0):.0f} tok/s)")
print(f"Output: {text!r}")


In [ ]:

# MLX baseline — only used for comparison, never imported in src/.
try:
    import mlx.core as mx
    # Trivial baseline: time a matmul of similar size.
    A = mx.random.normal((1024, 1024))
    B = mx.random.normal((1024, 1024))
    mx.eval(A @ B)  # warmup
    t0 = time.perf_counter()
    for _ in range(32):
        mx.eval(A @ B)
    t1 = time.perf_counter()
    print(f"MLX baseline: 32 matmuls in {(t1-t0)*1e3:.1f} ms "
          f"({(t1-t0)*1e3/32:.2f} ms/matmul)")
except Exception as e:
    print(f"MLX baseline skipped: {e}")


In [ ]:

nexusrt.firmware.shutdown()
print("✓ M1 Pro token pipeline test complete.")
